# Metode fundamentale de proiectare a algoritmilor

## Suport de seminar

### Cuprins
10. **Branch and Bound**: explorarea spatiului solutiilor cu limitare si eliminare a ramurilor nepromitatoare;
11. **Programare Dinamica**: memorarea subproblemelor si eliminarea recalcularilor.


## Legatura cu metodele anterioare

Metodele Branch and Bound si Programare Dinamica completeaza natural metodele Greedy, Divide et Impera si Backtracking.

---

### Backtracking
Backtracking exploreaza spatiul solutiilor si elimina ramurile care nu mai pot conduce la o solutie valida.

---

### Branch and Bound
Branch and Bound este asemanator cu Backtracking, dar merge mai departe:
- elimina ramurile invalide;
- elimina si ramurile valide care nu mai pot produce o solutie mai buna decat cea deja cunoscuta.

---

### Divide et Impera
Divide et Impera imparte problema in subprobleme independente.

---

### Programare Dinamica
Programarea dinamica apare atunci cand subproblemele nu sunt independente, ci se repeta.

Prin urmare:
- Divide et Impera descompune;
- Programarea Dinamica descompune si memoreaza.


# 10. Metoda Branch and Bound

## 10.1 Ideea de baza

Metoda Branch and Bound este o tehnica de optimizare folosita pentru probleme cu spatiu foarte mare al solutiilor.

Ideea centrala este:
- exploram sistematic spatiul solutiilor;
- impartim problema in subprobleme mai mici (**branch**);
- estimam cat de buna poate deveni fiecare subproblema (**bound**);
- eliminam ramurile care nu mai pot conduce la o solutie mai buna.

Metoda este apropiata de Backtracking, dar introduce o idee suplimentara: folosirea unor limite superioare sau inferioare pentru eliminarea ramurilor nepromitatoare.


## 10.2 Branch, Bound si Pruning

### Branch
Problema este impartita in subprobleme. Fiecare decizie genereaza noi ramuri.

### Bound
Pentru fiecare subproblema calculam o estimare a celei mai bune solutii care mai poate fi obtinuta din acea ramura.

### Pruning
Daca estimarea arata ca ramura nu poate depasi solutia optima deja cunoscuta, ramura este abandonata.

Astfel, algoritmul evita explorarea unor zone mari din spatiul solutiilor.


## 10.3 Model conceptual

Executia poate fi privita ca un arbore:
- fiecare nod reprezinta o solutie partiala;
- fiecare muchie reprezinta o alegere;
- fiecare frunza reprezinta o solutie completa sau o ramura respinsa.

Algoritmul:
1. extinde o ramura;
2. calculeaza limita;
3. decide daca merita continuata.


## 10.4 Exemplu intuitiv: problema rucsacului 0/1

### Enunt
Avem obiecte caracterizate prin:
- valoare;
- greutate.

Dorim sa maximizam valoarea totala fara a depasi capacitatea rucsacului.

In varianta 0/1, fiecare obiect poate fi luat integral sau deloc.

Pentru `n` obiecte exista:
```text
2^n
```
configuratii posibile.


## 10.5 Idee de bounding pentru Knapsack

Presupunem ca avem deja o valoare curenta si vrem sa estimam valoarea maxima posibila din continuarea ramurii.

O metoda clasica este:
- sortam obiectele dupa raportul valoare/greutate;
- completam optimist rucsacul ca in varianta fractionara;
- obtinem o limita superioara.

Daca aceasta limita este mai mica decat cea mai buna solutie deja gasita, ramura este abandonata.


In [1]:
class Node:
    def __init__(self, level, profit, weight, bound_value):
        self.level = level
        self.profit = profit
        self.weight = weight
        self.bound = bound_value


def bound(node, n, capacity, weights, values):
    if node.weight >= capacity:
        return 0

    result = node.profit
    j = node.level + 1
    total_weight = node.weight

    while j < n and total_weight + weights[j] <= capacity:
        total_weight += weights[j]
        result += values[j]
        j += 1

    if j < n:
        result += (capacity - total_weight) * values[j] / weights[j]

    return result


def knapsack_branch_bound(weights, values, capacity):
    n = len(weights)

    items = sorted(
        zip(values, weights),
        key=lambda x: x[0] / x[1],
        reverse=True
    )

    values = [x[0] for x in items]
    weights = [x[1] for x in items]

    queue = []
    root = Node(-1, 0, 0, 0)
    root.bound = bound(root, n, capacity, weights, values)
    queue.append(root)

    max_profit = 0

    while queue:
        node = queue.pop(0)

        if node.level == n - 1:
            continue

        level = node.level + 1

        # Cazul 1: luam obiectul curent
        new_weight = node.weight + weights[level]
        new_profit = node.profit + values[level]

        if new_weight <= capacity and new_profit > max_profit:
            max_profit = new_profit

        child_take = Node(level, new_profit, new_weight, 0)
        child_take.bound = bound(child_take, n, capacity, weights, values)

        if child_take.bound > max_profit:
            queue.append(child_take)

        # Cazul 2: nu luam obiectul curent
        child_skip = Node(level, node.profit, node.weight, 0)
        child_skip.bound = bound(child_skip, n, capacity, weights, values)

        if child_skip.bound > max_profit:
            queue.append(child_skip)

    return max_profit


weights = [2, 3, 4, 5]
values = [40, 50, 65, 70]
capacity = 8

print(knapsack_branch_bound(weights, values, capacity))


120


### Explicatie

Pentru fiecare nod:
- calculam valoarea actuala;
- estimam valoarea maxima posibila;
- continuam doar daca exista sansa unei solutii mai bune.

Astfel, multe combinatii nu mai sunt explorate.


## 10.6 Tipuri de explorare

### Depth First Search
Exploram adanc o ramura inainte de revenire. Are avantajul unui consum mai redus de memorie.

### Breadth First Search
Exploram nivel cu nivel. Poate gasi relativ repede solutii bune.

### Best First Search
Exploram mai intai nodurile cu limita cea mai promitatoare. Este o varianta frecvent folosita in probleme de optimizare.


## 10.7 Aplicatii clasice

Branch and Bound este folosit pentru:
- Traveling Salesman Problem;
- Knapsack 0/1;
- scheduling;
- integer programming;
- probleme combinatoriale de optimizare.


## 10.8 Avantaje si limite

### Avantaje
- garanteaza optimul global;
- reduce cautarea fata de brute force;
- este puternic pentru optimizare combinatoriala.

### Limite
- poate ramane exponential in cel mai rau caz;
- eficienta depinde de calitatea functiei de bound;
- implementarea poate deveni complexa.


## 10.9 Exercitii rezolvate

### Exercitiul 1
Explicati de ce Branch and Bound poate fi mai rapid decat brute force.

### Solutie
Pentru ca elimina ramuri care nu mai pot conduce la o solutie optima.

---

### Exercitiul 2
Ce rol are functia de bound?

### Solutie
Ea estimeaza cea mai buna valoare posibila care poate fi obtinuta dintr-o ramura.


## 10.10 Exercitii propuse

### Nivel introductiv
1. Explicati diferenta dintre Backtracking si Branch and Bound.
2. Desenati un arbore simplu de explorare.
3. Explicati rolul pruning-ului.

### Nivel intermediar
4. Implementati un Branch and Bound simplificat pentru Knapsack.
5. Comparati timpul cu brute force.
6. Implementati explorarea BFS.

### Nivel mai avansat
7. Studiati Traveling Salesman Problem.
8. Implementati Best First Search.
9. Proiectati o functie de bound mai buna pentru Knapsack.


## 10.11 Rezolvari pentru exercitiile propuse - Branch and Bound

### Exercitiul 1
Explicati diferenta dintre Backtracking si Branch and Bound.

### Rezolvare
Backtracking construieste solutia pas cu pas si elimina ramurile care devin invalide. Branch and Bound foloseste aceeasi idee de explorare a unui arbore al deciziilor, dar adauga o estimare numerica numita `bound`.

Daca estimarea arata ca o ramura nu mai poate conduce la o solutie mai buna decat solutia curenta, ramura este oprita chiar daca este inca valida.

Prin urmare:
- Backtracking elimina ramuri imposibile;
- Branch and Bound elimina ramuri imposibile si ramuri nepromitatoare.

---

### Exercitiul 2
Desenati un arbore simplu de explorare.

### Rezolvare
Consideram o problema cu doua decizii binare: la fiecare pas alegem `0` sau `1`.

```text
              []
            /    \
          [0]    [1]
         /  \    /  \
      [0,0][0,1][1,0][1,1]
```

Fiecare nod reprezinta o solutie partiala. Frunzele reprezinta solutii complete. In Branch and Bound, fiecare nod poate avea si o valoare estimata:

```text
nod = solutie partiala + cost curent + bound
```

---

### Exercitiul 3
Explicati rolul pruning-ului.

### Rezolvare
Pruning-ul inseamna eliminarea unei ramuri din arborele de cautare.

In Branch and Bound, o ramura este eliminata daca:
- depaseste restrictiile problemei;
- are un bound mai slab decat solutia deja cunoscuta;
- nu mai poate conduce la optim.

Acest lucru reduce numarul de noduri explorate si face algoritmul mai eficient in practica.

---

### Exercitiul 4
Implementati un Branch and Bound simplificat pentru Knapsack.

### Rezolvare
Fiecare nod contine nivelul, profitul curent, greutatea curenta si limita superioara estimata. Bound-ul este calculat optimist prin completarea rucsacului cu fractii de obiecte.


In [ ]:
from collections import deque

class Node:
    def __init__(self, level, profit, weight, bound_value=0):
        self.level = level
        self.profit = profit
        self.weight = weight
        self.bound = bound_value


def compute_bound(node, n, capacity, weights, values):
    if node.weight >= capacity:
        return 0

    result = node.profit
    total_weight = node.weight
    j = node.level + 1

    while j < n and total_weight + weights[j] <= capacity:
        total_weight += weights[j]
        result += values[j]
        j += 1

    if j < n:
        result += (capacity - total_weight) * values[j] / weights[j]

    return result


def knapsack_branch_bound(weights, values, capacity):
    n = len(weights)

    items = sorted(zip(values, weights), key=lambda x: x[0] / x[1], reverse=True)
    values = [v for v, w in items]
    weights = [w for v, w in items]

    root = Node(level=-1, profit=0, weight=0)
    root.bound = compute_bound(root, n, capacity, weights, values)

    queue = deque([root])
    max_profit = 0

    while queue:
        node = queue.popleft()

        if node.level == n - 1:
            continue

        next_level = node.level + 1

        # Cazul 1: luam obiectul curent.
        take_node = Node(
            next_level,
            node.profit + values[next_level],
            node.weight + weights[next_level]
        )

        if take_node.weight <= capacity and take_node.profit > max_profit:
            max_profit = take_node.profit

        take_node.bound = compute_bound(take_node, n, capacity, weights, values)
        if take_node.bound > max_profit:
            queue.append(take_node)

        # Cazul 2: nu luam obiectul curent.
        skip_node = Node(next_level, node.profit, node.weight)
        skip_node.bound = compute_bound(skip_node, n, capacity, weights, values)

        if skip_node.bound > max_profit:
            queue.append(skip_node)

    return max_profit


weights = [2, 3, 4, 5]
values = [40, 50, 65, 70]
capacity = 8

print(knapsack_branch_bound(weights, values, capacity))


### Exercitiul 5
Comparati timpul cu brute force.

### Rezolvare
Brute force verifica toate submultimile. Branch and Bound evita explorarea unor ramuri atunci cand bound-ul lor nu mai poate depasi solutia curenta.


In [2]:
import time


def knapsack_bruteforce(weights, values, capacity):
    n = len(weights)
    best = 0

    for mask in range(1 << n):
        total_w = 0
        total_v = 0

        for i in range(n):
            if mask & (1 << i):
                total_w += weights[i]
                total_v += values[i]

        if total_w <= capacity:
            best = max(best, total_v)

    return best


weights = [2, 3, 4, 5, 9, 7, 6, 4, 3, 8]
values = [40, 50, 65, 70, 120, 90, 80, 45, 30, 100]
capacity = 20

start = time.perf_counter()
result_bf = knapsack_bruteforce(weights, values, capacity)
time_bf = time.perf_counter() - start

start = time.perf_counter()
result_bb = knapsack_branch_bound(weights, values, capacity)
time_bb = time.perf_counter() - start

print("Brute force:", result_bf, "timp:", time_bf)
print("Branch and Bound:", result_bb, "timp:", time_bb)


Brute force: 305 timp: 0.0009999999999763531
Branch and Bound: 305 timp: 0.00010000000008858478


### Exercitiul 6
Implementati explorarea BFS.

### Rezolvare
BFS exploreaza arborele nivel cu nivel. In exemplul urmator generam toate sirurile binare de lungime `n` folosind o coada.


In [ ]:
from collections import deque


def bfs_binary_strings(n):
    queue = deque([[]])
    result = []

    while queue:
        current = queue.popleft()

        if len(current) == n:
            result.append(current)
        else:
            queue.append(current + [0])
            queue.append(current + [1])

    return result


print(bfs_binary_strings(3))


### Exercitiul 7
Studiati Traveling Salesman Problem.

### Rezolvare
Traveling Salesman Problem cere gasirea unui traseu de cost minim care porneste dintr-un oras, viziteaza fiecare oras exact o data si revine la orasul initial.

Pentru `n` orase, numarul de trasee posibile este aproximativ `(n-1)!`. De aceea problema devine rapid dificila. Mai jos este o solutie exhaustiva pentru un numar mic de orase.


In [ ]:
from itertools import permutations


def tsp_bruteforce(cost):
    n = len(cost)
    cities = list(range(1, n))
    best_cost = float('inf')
    best_route = None

    for perm in permutations(cities):
        route = [0] + list(perm) + [0]
        total = 0

        for i in range(len(route) - 1):
            total += cost[route[i]][route[i + 1]]

        if total < best_cost:
            best_cost = total
            best_route = route

    return best_cost, best_route


cost = [
    [0, 10, 15, 20],
    [10, 0, 35, 25],
    [15, 35, 0, 30],
    [20, 25, 30, 0]
]

print(tsp_bruteforce(cost))


### Exercitiul 8
Implementati Best First Search.

### Rezolvare
Best First Search alege mai intai nodul considerat cel mai promitator. Pentru aceasta folosim o coada de prioritati. In exemplul urmator alegem mai intai nodul cu bound mai mare.


In [ ]:
import heapq


def knapsack_best_first(weights, values, capacity):
    n = len(weights)
    items = sorted(zip(values, weights), key=lambda x: x[0] / x[1], reverse=True)
    values = [v for v, w in items]
    weights = [w for v, w in items]

    root = Node(-1, 0, 0)
    root.bound = compute_bound(root, n, capacity, weights, values)

    heap = []
    counter = 0
    heapq.heappush(heap, (-root.bound, counter, root))

    max_profit = 0

    while heap:
        _, _, node = heapq.heappop(heap)

        if node.bound <= max_profit or node.level == n - 1:
            continue

        next_level = node.level + 1

        take_node = Node(
            next_level,
            node.profit + values[next_level],
            node.weight + weights[next_level]
        )

        if take_node.weight <= capacity and take_node.profit > max_profit:
            max_profit = take_node.profit

        take_node.bound = compute_bound(take_node, n, capacity, weights, values)
        if take_node.bound > max_profit:
            counter += 1
            heapq.heappush(heap, (-take_node.bound, counter, take_node))

        skip_node = Node(next_level, node.profit, node.weight)
        skip_node.bound = compute_bound(skip_node, n, capacity, weights, values)

        if skip_node.bound > max_profit:
            counter += 1
            heapq.heappush(heap, (-skip_node.bound, counter, skip_node))

    return max_profit


weights = [2, 3, 4, 5]
values = [40, 50, 65, 70]
capacity = 8

print(knapsack_best_first(weights, values, capacity))


### Exercitiul 9
Proiectati o functie de bound mai buna pentru Knapsack.

### Rezolvare
Un bound mai bun trebuie sa fie apropiat de valoarea reala maxima, dar rapid de calculat. O varianta buna este bound-ul fractional: sortam obiectele dupa raportul valoare/greutate, completam capacitatea ramasa cu obiecte intregi si apoi cu o fractie din urmatorul obiect.


In [ ]:
def fractional_upper_bound(level, profit, weight, weights, values, capacity):
    if weight >= capacity:
        return 0

    n = len(weights)
    bound_value = profit
    total_weight = weight
    j = level + 1

    while j < n and total_weight + weights[j] <= capacity:
        total_weight += weights[j]
        bound_value += values[j]
        j += 1

    if j < n:
        remaining_capacity = capacity - total_weight
        bound_value += remaining_capacity * values[j] / weights[j]

    return bound_value


weights = [2, 3, 4, 5]
values = [40, 50, 65, 70]
capacity = 8

items = sorted(zip(values, weights), key=lambda x: x[0] / x[1], reverse=True)
values_sorted = [v for v, w in items]
weights_sorted = [w for v, w in items]

print(fractional_upper_bound(-1, 0, 0, weights_sorted, values_sorted, capacity))


# 11. Metoda Programarii Dinamice

## 11.1 Ideea de baza

Programarea dinamica este o tehnica de optimizare folosita atunci cand:
- problema are subprobleme repetitive;
- aceleasi calcule apar de multe ori.

Ideea centrala este:
- calculam o singura data fiecare subproblema;
- memoram rezultatul;
- il reutilizam ulterior.

Astfel evitam recalcularile inutile.


## 11.2 Proprietati esentiale

Pentru ca o problema sa poata fi rezolvata prin programare dinamica, apar de regula doua proprietati.

### 1. Substructura optima
Solutia optima poate fi construita din solutii optime ale unor subprobleme.

### 2. Subprobleme suprapuse
Aceeasi subproblema apare de multe ori.

Aceasta este diferenta esentiala fata de Divide et Impera.


## 11.3 Strategii principale

### Top-Down: Memoization
- folosim recursivitate;
- memoram rezultatele deja calculate.

### Bottom-Up: Tabulation
- construim iterativ solutia;
- pornim de la cazurile mici.


## 11.4 Exemplu clasic: Fibonacci

Formula este:
```text
F(n) = F(n-1) + F(n-2)
```

Varianta recursiva simpla recalculeaza de multe ori aceleasi valori.


In [ ]:
def fib_naiv(n):
    if n <= 1:
        return n

    return fib_naiv(n - 1) + fib_naiv(n - 2)


print(fib_naiv(10))


### Complexitate

Varianta recursiva naiva are complexitate aproximativ exponentiala:
```text
O(2^n)
```


In [ ]:
def fib_memo(n, memo=None):
    if memo is None:
        memo = {}

    if n in memo:
        return memo[n]

    if n <= 1:
        return n

    memo[n] = fib_memo(n - 1, memo) + fib_memo(n - 2, memo)
    return memo[n]


print(fib_memo(10))


### Complexitate

Fiecare valoare este calculata o singura data:
```text
O(n)
```


In [ ]:
def fib_dp(n):
    if n <= 1:
        return n

    dp = [0] * (n + 1)
    dp[1] = 1

    for i in range(2, n + 1):
        dp[i] = dp[i - 1] + dp[i - 2]

    return dp[n]


print(fib_dp(10))


## 11.5 Cum construim o solutie DP?

De regula urmam pasii:
1. Definim starea.
2. Identificam tranzitia.
3. Stabilim cazurile de baza.
4. Alegem ordinea de calcul.
5. Calculam raspunsul final.


## 11.6 Exemplu rezolvat: Coin Change

### Enunt
Avem monede cu valori date si dorim sa obtinem o suma folosind un numar minim de monede.

### Stare
```text
dp[i] = numarul minim de monede pentru suma i
```

### Tranzitie
```text
dp[i] = min(dp[i], dp[i-coin] + 1)
```


In [ ]:
def coin_change(coins, amount):
    INF = float('inf')

    dp = [INF] * (amount + 1)
    dp[0] = 0

    for i in range(1, amount + 1):
        for coin in coins:
            if coin <= i:
                dp[i] = min(dp[i], dp[i - coin] + 1)

    return dp[amount]


coins = [1, 3, 4]
print(coin_change(coins, 6))


### Observatie

Pentru monedele `[1, 3, 4]` si suma `6`, Greedy poate esua, dar programarea dinamica gaseste solutia optima:
```text
3 + 3
```


## 11.7 Exemplu rezolvat: Longest Common Subsequence

### Enunt
Avem doua siruri si dorim lungimea celui mai lung subsir comun.

Exemplu:
```text
ABCBDAB
BDCABA
```

### Stare
```text
dp[i][j]
```
reprezinta lungimea celui mai lung subsir comun pentru prefixele `s1[0..i-1]` si `s2[0..j-1]`.


### Tranzitie

Daca ultimele caractere sunt egale:
```text
dp[i][j] = dp[i-1][j-1] + 1
```

Altfel:
```text
dp[i][j] = max(dp[i-1][j], dp[i][j-1])
```


In [ ]:
def lcs(s1, s2):
    n = len(s1)
    m = len(s2)

    dp = [[0] * (m + 1) for _ in range(n + 1)]

    for i in range(1, n + 1):
        for j in range(1, m + 1):
            if s1[i - 1] == s2[j - 1]:
                dp[i][j] = dp[i - 1][j - 1] + 1
            else:
                dp[i][j] = max(dp[i - 1][j], dp[i][j - 1])

    return dp[n][m]


print(lcs("ABCBDAB", "BDCABA"))


## 11.8 Exemplu rezolvat: Knapsack 0/1 cu DP

### Stare
```text
dp[i][w]
```
reprezinta valoarea maxima obtinuta folosind primele `i` obiecte si capacitatea `w`.

### Tranzitie
Pentru fiecare obiect exista doua variante:
- nu il luam;
- il luam, daca incape.


In [ ]:
def knapsack_dp(weights, values, capacity):
    n = len(weights)

    dp = [[0] * (capacity + 1) for _ in range(n + 1)]

    for i in range(1, n + 1):
        for w in range(capacity + 1):
            dp[i][w] = dp[i - 1][w]

            if weights[i - 1] <= w:
                dp[i][w] = max(
                    dp[i][w],
                    dp[i - 1][w - weights[i - 1]] + values[i - 1]
                )

    return dp[n][capacity]


weights = [2, 3, 4, 5]
values = [40, 50, 65, 70]
capacity = 8

print(knapsack_dp(weights, values, capacity))


### Complexitate

Pentru Knapsack 0/1 cu programare dinamica:
```text
O(n * capacity)
```

Aceasta poate fi mult mai buna decat explorarea celor `2^n` combinatii.


## 11.9 Diferenta dintre DP si Divide et Impera

### Divide et Impera
- subprobleme independente;
- putine recalculari.

### Programare Dinamica
- subproblemele se suprapun;
- rezultatele sunt memorate si reutilizate.


## 11.10 Diferenta dintre DP si Greedy

### Greedy
- face alegeri locale;
- nu garanteaza mereu optimul global.

### Programare Dinamica
- exploreaza sistematic posibilitatile relevante;
- garanteaza optimul pentru problemele care respecta proprietatile DP.


## 11.11 Avantaje si limite

### Avantaje
- evita recalcularile;
- garanteaza optimul pentru multe probleme;
- poate transforma probleme exponentiale in probleme polinomiale.

### Limite
- definirea starilor poate fi dificila;
- poate consuma multa memorie;
- unele probleme au prea multe stari.


## 11.12 Aplicatii clasice

Programarea dinamica apare frecvent in:
- edit distance;
- longest common subsequence;
- shortest paths;
- knapsack;
- bioinformatica;
- optimizare combinatoriala.


## 11.13 Exercitii rezolvate

### Exercitiul 1
De ce Fibonacci recursiv este ineficient?

### Solutie
Pentru ca recalculeaza de foarte multe ori aceleasi subprobleme.

---

### Exercitiul 2
Care este diferenta dintre memoization si tabulation?

### Solutie
- memoization = top-down;
- tabulation = bottom-up.


## 11.14 Exercitii propuse

### Nivel introductiv
1. Implementati Fibonacci Bottom-Up.
2. Calculati manual tabelul DP pentru Coin Change.
3. Explicati subproblemele suprapuse.

### Nivel intermediar
4. Implementati Longest Common Subsequence.
5. Rezolvati Knapsack 0/1.
6. Optimizati memoria pentru Fibonacci.

### Nivel mai avansat
7. Studiati Floyd-Warshall.
8. Implementati Edit Distance.
9. Comparati Greedy si DP pentru Coin Change.


## 11.15 Rezolvari pentru exercitiile propuse - Programare Dinamica

### Exercitiul 1
Implementati Fibonacci Bottom-Up.

### Rezolvare
Construim valorile in ordine crescatoare. Pornim de la cazurile de baza `F(0)=0` si `F(1)=1`, apoi calculam fiecare termen folosind cei doi termeni anteriori.


In [ ]:
def fibonacci_bottom_up(n):
    if n <= 1:
        return n

    dp = [0] * (n + 1)
    dp[1] = 1

    for i in range(2, n + 1):
        dp[i] = dp[i - 1] + dp[i - 2]

    return dp[n]


print(fibonacci_bottom_up(10))


### Exercitiul 2
Calculati manual tabelul DP pentru Coin Change.

### Rezolvare
Pentru monedele `[1, 3, 4]` si suma `6`, definim `dp[i]` ca numarul minim de monede necesare pentru suma `i`. Cazul de baza este `dp[0] = 0`.


In [ ]:
def coin_change_table(coins, amount):
    INF = float('inf')
    dp = [INF] * (amount + 1)
    dp[0] = 0

    rows = []
    for i in range(1, amount + 1):
        for coin in coins:
            if coin <= i:
                dp[i] = min(dp[i], dp[i - coin] + 1)
        rows.append((i, dp[i] if dp[i] != INF else None))

    return dp, rows


coins = [1, 3, 4]
amount = 6

dp, rows = coin_change_table(coins, amount)
print("Tabel dp:")
for suma, minim in rows:
    print(f"suma {suma}: {minim}")

print("Raspuns final:", dp[amount])


### Exercitiul 3
Explicati subproblemele suprapuse.

### Rezolvare
Subproblemele suprapuse apar atunci cand aceeasi subproblema este calculata de mai multe ori.

Exemplu: in calculul recursiv naiv pentru Fibonacci, `F(3)` poate fi recalculat in ramuri diferite ale arborelui de apeluri. Programarea dinamica evita acest lucru memorand rezultatul.


In [ ]:
call_count = {}


def fib_count(n):
    call_count[n] = call_count.get(n, 0) + 1

    if n <= 1:
        return n

    return fib_count(n - 1) + fib_count(n - 2)


fib_count(6)
print(call_count)


### Exercitiul 4
Implementati Longest Common Subsequence.

### Rezolvare
Definim `dp[i][j]` ca lungimea celui mai lung subsir comun pentru prefixele `s1[:i]` si `s2[:j]`.

Daca ultimele caractere sunt egale, extindem solutia anterioara. Altfel, pastram maximum dintre cele doua posibilitati: eliminam ultimul caracter din primul sir sau din al doilea sir.


In [ ]:
def lcs_length(s1, s2):
    n = len(s1)
    m = len(s2)

    dp = [[0] * (m + 1) for _ in range(n + 1)]

    for i in range(1, n + 1):
        for j in range(1, m + 1):
            if s1[i - 1] == s2[j - 1]:
                dp[i][j] = dp[i - 1][j - 1] + 1
            else:
                dp[i][j] = max(dp[i - 1][j], dp[i][j - 1])

    return dp[n][m]


print(lcs_length("ABCBDAB", "BDCABA"))


### Exercitiul 5
Rezolvati Knapsack 0/1.

### Rezolvare
Definim `dp[i][w]` ca valoarea maxima obtinuta folosind primele `i` obiecte si o capacitate maxima `w`.

Pentru fiecare obiect avem doua optiuni:
- nu il luam;
- il luam, daca incape.


In [ ]:
def knapsack_dp(weights, values, capacity):
    n = len(weights)
    dp = [[0] * (capacity + 1) for _ in range(n + 1)]

    for i in range(1, n + 1):
        for w in range(capacity + 1):
            dp[i][w] = dp[i - 1][w]

            if weights[i - 1] <= w:
                dp[i][w] = max(
                    dp[i][w],
                    dp[i - 1][w - weights[i - 1]] + values[i - 1]
                )

    return dp[n][capacity]


weights = [2, 3, 4, 5]
values = [40, 50, 65, 70]
capacity = 8

print(knapsack_dp(weights, values, capacity))


### Exercitiul 6
Optimizati memoria pentru Fibonacci.

### Rezolvare
Pentru calculul lui Fibonacci nu este necesar sa memoram tot vectorul `dp`. Avem nevoie doar de ultimele doua valori.


In [ ]:
def fibonacci_optimized(n):
    if n <= 1:
        return n

    prev2 = 0
    prev1 = 1

    for _ in range(2, n + 1):
        current = prev1 + prev2
        prev2 = prev1
        prev1 = current

    return prev1


print(fibonacci_optimized(10))


### Exercitiul 7
Studiati Floyd-Warshall.

### Rezolvare
Floyd-Warshall calculeaza cele mai scurte drumuri intre toate perechile de noduri dintr-un graf ponderat.

Starea este `dist[i][j]`, adica distanta minima cunoscuta de la nodul `i` la nodul `j`.

Tranzitia verifica daca un drum printr-un nod intermediar `k` este mai bun:

```text
dist[i][j] = min(dist[i][j], dist[i][k] + dist[k][j])
```


In [ ]:
def floyd_warshall(graph):
    n = len(graph)
    dist = [row[:] for row in graph]

    for k in range(n):
        for i in range(n):
            for j in range(n):
                if dist[i][k] + dist[k][j] < dist[i][j]:
                    dist[i][j] = dist[i][k] + dist[k][j]

    return dist


INF = float('inf')
graph = [
    [0, 3, INF, 7],
    [8, 0, 2, INF],
    [5, INF, 0, 1],
    [2, INF, INF, 0]
]

for row in floyd_warshall(graph):
    print(row)


### Exercitiul 8
Implementati Edit Distance.

### Rezolvare
Edit Distance masoara numarul minim de operatii necesare pentru a transforma un sir in altul. Operatiile uzuale sunt:
- inserare;
- stergere;
- inlocuire.

Definim `dp[i][j]` ca distanta minima dintre prefixele `s1[:i]` si `s2[:j]`.


In [ ]:
def edit_distance(s1, s2):
    n = len(s1)
    m = len(s2)

    dp = [[0] * (m + 1) for _ in range(n + 1)]

    for i in range(n + 1):
        dp[i][0] = i

    for j in range(m + 1):
        dp[0][j] = j

    for i in range(1, n + 1):
        for j in range(1, m + 1):
            if s1[i - 1] == s2[j - 1]:
                dp[i][j] = dp[i - 1][j - 1]
            else:
                dp[i][j] = 1 + min(
                    dp[i - 1][j],      # stergere
                    dp[i][j - 1],      # inserare
                    dp[i - 1][j - 1]   # inlocuire
                )

    return dp[n][m]


print(edit_distance("kitten", "sitting"))


### Exercitiul 9
Comparati Greedy si DP pentru Coin Change.

### Rezolvare
Pentru monedele `[1, 3, 4]` si suma `6`, metoda Greedy alege mai intai moneda `4`, apoi doua monede `1`. Rezultatul are 3 monede.

Programarea dinamica gaseste solutia optima: `3 + 3`, adica 2 monede.


In [ ]:
def greedy_coin_change(amount, coins):
    coins = sorted(coins, reverse=True)
    result = []

    for coin in coins:
        while amount >= coin:
            result.append(coin)
            amount -= coin

    return result


def dp_coin_change_min(coins, amount):
    INF = float('inf')
    dp = [INF] * (amount + 1)
    parent = [-1] * (amount + 1)
    dp[0] = 0

    for i in range(1, amount + 1):
        for coin in coins:
            if coin <= i and dp[i - coin] + 1 < dp[i]:
                dp[i] = dp[i - coin] + 1
                parent[i] = coin

    solution = []
    current = amount
    while current > 0 and parent[current] != -1:
        solution.append(parent[current])
        current -= parent[current]

    return dp[amount], solution


coins = [1, 3, 4]
amount = 6

print("Greedy:", greedy_coin_change(amount, coins))
print("DP:", dp_coin_change_min(coins, amount))


# Concluzii

## Branch and Bound
- exploreaza spatiul solutiilor;
- foloseste limite pentru pruning;
- garanteaza optimul global;
- este important pentru optimizare combinatoriala.

## Programare Dinamica
- evita recalcularile;
- memoreaza subproblemele;
- transforma multe probleme dificile in algoritmi eficienti.
